# Stock EDA & Modeling Notebook

This notebook reproduces the EDA, feature engineering, modeling, and evaluation steps we ran earlier.
It expects the file `combined.csv` to be present in the same environment (path: `/mnt/data/combined.csv`).

**Contents**
- Load data and quick inspection
- Feature engineering (shifted surprise, shifted rep eps, technical features)
- Chronological train/test split
- Pipelines for Logistic Regression, RandomForest, and GradientBoosting
- Threshold tuning (Youden's J and F1 grid)
- Outlier experiment and per-sector model comparison
- Save models and preprocessing pipelines (optional)

Run cells sequentially.

In [ ]:
# Cell: Load libraries and data
import pandas as pd, numpy as np, os
from datetime import datetime
print("cwd:", os.getcwd())
df = pd.read_csv("/mnt/data/combined.csv")
print("Loaded dataframe shape:", df.shape)
df.head()

In [ ]:
# Cell: Basic preprocessing & feature engineering
# Convert dates
for col in ["Date_x","Date_y","Earnings Date","next_trading_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Ensure target numeric and drop NA target rows
df["daily_change_y"] = pd.to_numeric(df["daily_change_y"], errors="coerce")
df = df.dropna(subset=["daily_change_y"]).copy()

# Shift Surprise(%) by 1 day per ticker to avoid leakage
if "Surprise(%)" in df.columns:
    df["Surprise(%)"] = pd.to_numeric(df["Surprise(%)"], errors="coerce")
    df = df.sort_values(["ticker","Date_x"])
    df["surprise_shift1"] = df.groupby("ticker")["Surprise(%)"].shift(1)

# Drop leaky columns
for col in ["Reported EPS","Surprise(%)","Adj Close"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# New engineered features
df["close_open_return"] = (df["Close"] - df["Open"]) / df["Open"]
df["high_low_range"] = (df["High"] - df["Low"]) / df["Open"]
if "sma20" in df.columns:
    df["close_sma20_ratio"] = df["Close"] / df["sma20"]
df["vol_log"] = np.log1p(df["Volume"].fillna(0))

# Binary target for 0.03
df["y_030"] = (df["daily_change_y"] >= 0.03).astype(int)

print("After preprocessing shape:", df.shape)
df[[ "ticker","Date_x","daily_change_y","y_030","surprise_shift1","shifted_rep_eps","close_open_return"]].head()

In [ ]:
# Cell: Train/test split, preprocessing, and modeling
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score, precision_score, recall_score

# Chronological split: first 80% train, last 20% test
df = df.sort_values("Date_x")
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()
print("Train / Test sizes:", train.shape, test.shape)

# Select features
numeric_feats = ["Open","High","Low","Close","Volume","close_open_return","high_low_range","vol_log",
                 "sma20","sma50","sma200","sma20_change","sma50_change","sma200_change","rsi_14",
                 "volatility_wk","volatility_1m","1w_change","1m_change","3m_change","1yr_change",
                 "TTM_EPS","PE_Ratio","Forward_PE","close_sma20_ratio"]
numeric_feats = [f for f in numeric_feats if f in df.columns]
cat_feats = ["Sector"] if "Sector" in df.columns else []

# Preprocessor
numeric_transform = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_transform = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse=False))])

preprocessor = ColumnTransformer([("num", numeric_transform, numeric_feats),
                                  ("cat", cat_transform, cat_feats)])

# Models and training
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=100, n_jobs=-1, class_weight="balanced"),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100)
}

X_train = train[numeric_feats + cat_feats]
y_train = train["y_030"]
X_test = test[numeric_feats + cat_feats]
y_test = test["y_030"]

trained = {}
results = {}
for name, model in models.items():
    pipe = Pipeline([("pre", preprocessor), ("clf", model)])
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:,1]
    pred = (prob >= 0.5).astype(int)
    roc = roc_auc_score(y_test, prob)
    precision, recall, _ = precision_recall_curve(y_test, prob)
    pr_auc = auc(recall, precision)
    f1 = f1_score(y_test, pred)
    results[name] = {"roc_auc": roc, "pr_auc": pr_auc, "f1_0.5": f1}
    trained[name] = pipe

results

In [ ]:
# Cell: Threshold tuning function and apply to models
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, matthews_corrcoef, roc_curve, precision_recall_curve

def find_best_thresholds(y_true, probs):
    fpr, tpr, thr = roc_curve(y_true, probs)
    youden_idx = np.argmax(tpr - fpr)
    youden_thr = thr[youden_idx]
    # Grid search for best F1
    grid = np.linspace(0,1,101)
    f1_grid = [f1_score(y_true, (probs>=g).astype(int)) for g in grid]
    best_f1_thr = grid[int(np.argmax(f1_grid))]
    return {"youden_thr": youden_thr, "best_f1_thr": best_f1_thr, "max_f1": max(f1_grid)}

threshold_results = {}
detailed_results = {}
for name, pipe in trained.items():
    prob = pipe.predict_proba(X_test)[:,1]
    thr_info = find_best_thresholds(y_test.values, prob)
    for key, thr in [("youden", thr_info["youden_thr"]), ("f1", thr_info["best_f1_thr"])]:
        preds = (prob >= thr).astype(int)
        detailed_results.setdefault(name, {})[key] = {
            "threshold": float(thr),
            "f1": float(f1_score(y_test, preds)),
            "precision": float(precision_score(y_test, preds, zero_division=0)),
            "recall": float(recall_score(y_test, preds, zero_division=0)),
            "mcc": float(matthews_corrcoef(y_test, preds))
        }
    threshold_results[name] = thr_info

threshold_results, detailed_results

In [ ]:
# Cell: Per-sector model comparison (for sectors with reasonable sample sizes)
sector_counts = df["Sector"].value_counts(dropna=True)
big_sectors = sector_counts[sector_counts >= 500].index.tolist()

sector_results = {}
for sec in big_sectors:
    train_sec = train[train["Sector"] == sec]
    test_sec = test[test["Sector"] == sec]
    if len(train_sec) < 50 or len(test_sec) < 30:
        continue
    X_tr = train_sec[numeric_feats + cat_feats]
    y_tr = train_sec["y_030"]
    X_te = test_sec[numeric_feats + cat_feats]
    y_te = test_sec["y_030"]
    pipe = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    pipe.fit(X_tr, y_tr)
    prob = pipe.predict_proba(X_te)[:,1]
    roc = roc_auc_score(y_te, prob)
    sector_results[sec] = {"train_n": len(train_sec), "test_n": len(test_sec), "roc_auc": roc}

sector_results

In [ ]:
# Cell: Save notebook artifacts: example save of the logistic pipeline and feature list
import joblib, os
os.makedirs("/mnt/data/artifacts", exist_ok=True)
joblib.dump(trained["LogisticRegression"], "/mnt/data/artifacts/logistic_pipeline.joblib")
joblib.dump(preprocessor, "/mnt/data/artifacts/preprocessor.joblib")
with open("/mnt/data/artifacts/feature_list.txt","w") as f:
    f.write("\n".join(['Open', 'High', 'Low', 'Close', 'Volume', 'close_open_return', 'high_low_range', 'vol_log', 'sma20', 'sma50', 'sma200', 'sma20_change', 'sma50_change', 'sma200_change', 'rsi_14', 'volatility_wk', 'volatility_1m', '1w_change', '1m_change', '3m_change', '1yr_change', 'TTM_EPS', 'PE_Ratio', 'Forward_PE', 'close_sma20_ratio', 'Sector']))
print("Saved artifacts to /mnt/data/artifacts")